In [ ]:
import numpy as np
import torch
from sbi import GRUEstimator, train, evaluate, compare_models, heatmap_2d, BetaProposal, simulate_ar1_batch
from scipy import stats

# Problem config
A = 5.0
T = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
class AR1Estimator(GRUEstimator):
    def alter_outputs(self, x):
        rho = torch.tanh(x[:, 0])
        sigma = torch.exp(x[:, 1])
        return torch.stack([rho, sigma], dim=1)

In [ ]:
def sample_uniform(batch_size):
    rho = torch.empty(batch_size).uniform_(-1, 1)
    sigma = torch.empty(batch_size).uniform_(0, A)
    x = simulate_ar1_batch(rho, sigma, T)
    theta = torch.stack([rho, sigma], dim=1)
    return theta, x

In [ ]:
model_base = AR1Estimator()
model_base, history_base = train(model_base, sample_uniform, n_epochs=128, n_batches=128, lr=2e-3)
evaluate(model_base, sample_uniform, history_base, [r'$\rho$', r'$\sigma$'])

In [ ]:
def sample_is(batch_size, alpha_rho=0.5, alpha_sigma=0.5):
    rho_tilde = np.random.beta(alpha_rho, alpha_rho, size=batch_size)
    sigma_tilde = np.random.beta(alpha_sigma, 1.0, size=batch_size)
    rho = torch.tensor(2 * rho_tilde - 1, dtype=torch.float32)
    sigma = torch.tensor(A * sigma_tilde, dtype=torch.float32)
    x = simulate_ar1_batch(rho, sigma, T)
    q_rho = stats.beta.pdf(rho_tilde, alpha_rho, alpha_rho)
    q_sigma = stats.beta.pdf(sigma_tilde, alpha_sigma, 1.0)
    weights = 1.0 / (q_rho * q_sigma)
    theta = torch.stack([rho, sigma], dim=1)
    return theta, x, torch.tensor(weights, dtype=torch.float32)

In [ ]:
models = {'Baseline': model_base}
histories = {'Baseline': history_base}

for alpha in [0.7, 0.5, 0.3]:
    name = f'IS a={alpha}'
    fn = lambda bs, a=alpha: sample_is(bs, alpha_rho=a, alpha_sigma=a)
    m = AR1Estimator()
    m, h = train(m, fn, n_epochs=128, n_batches=128, lr=2e-3, seed=0)
    models[name] = m
    histories[name] = h

In [ ]:
test_set = sample_uniform(8192)
compare_models(models, histories, test_set, [r'$\rho$', r'$\sigma$'],
               param_ranges=[(-1, 1), (0, A)])

In [ ]:
# 2D MSE heatmaps
theta_true = test_set[0].numpy()
preds = {}
for name, model in models.items():
    model.eval()
    with torch.no_grad():
        d = next(model.parameters()).device
        preds[name] = model(test_set[1].to(d)).cpu().numpy()

heatmap_2d(theta_true, preds, [r'$\rho$', r'$\sigma$'],
           param_ranges=[(-1, 1), (0, A)])